# License Plate Detection — Faster R-CNN (Google Colab)
**CMPS 261 — Machine Learning Project**

Run this on Colab with a **T4 GPU**.

**Before running:** Upload `license_plate_data.zip` when prompted.

## 1. Install Dependencies

In [ ]:
!pip install pycocotools -q

## 2. Upload & Extract Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload license_plate_data.zip

In [ ]:
import zipfile, os

with zipfile.ZipFile('license_plate_data.zip', 'r') as z:
    z.extractall('/content/')

print('Extracted successfully.')

## 3. Dataset Class

In [ ]:
import os
import xml.etree.ElementTree as ET
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms.functional as F

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, ann_dir):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.samples = sorted([f for f in os.listdir(ann_dir) if f.endswith('.xml')])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        xml_file = self.samples[idx]
        tree = ET.parse(os.path.join(self.ann_dir, xml_file))
        root = tree.getroot()
        filename = root.find('filename').text
        img = Image.open(os.path.join(self.img_dir, filename)).convert('RGB')
        boxes = []
        for obj in root.findall('object'):
            xmin = int(obj.find('bndbox/xmin').text)
            ymin = int(obj.find('bndbox/ymin').text)
            xmax = int(obj.find('bndbox/xmax').text)
            ymax = int(obj.find('bndbox/ymax').text)
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        target = {
            'boxes'   : boxes,
            'labels'  : labels,
            'image_id': torch.tensor([idx]),
            'area'    : (boxes[:,3]-boxes[:,1]) * (boxes[:,2]-boxes[:,0]) if len(boxes) else torch.tensor([]),
            'iscrowd' : torch.zeros(len(boxes), dtype=torch.int64),
        }
        return F.to_tensor(img), target

def collate_fn(batch):
    return tuple(zip(*batch))

print('Dataset class ready.')

## 4. Load Data

In [ ]:
from torch.utils.data import DataLoader, random_split

IMG_DIR = '/content/data/yolo/images'
ANN_DIR = '/content/data/archive/annotations' if os.path.exists('/content/data/archive') else None

# Use the original archive annotations
full_dataset = LicensePlateDataset(
    '/content/data/yolo/images/train',
    '/content/data/yolo/labels/train'
)

# Re-create from archive if available
import glob
all_xmls = glob.glob('/content/data/**/*.xml', recursive=True)
print(f'Found {len(all_xmls)} XML files')

ann_dir = os.path.dirname(all_xmls[0]) if all_xmls else None
img_dir = ann_dir.replace('annotations', 'images') if ann_dir else None

full_dataset = LicensePlateDataset(img_dir, ann_dir)
n = len(full_dataset)
n_train, n_val = int(0.70*n), int(0.15*n)
n_test = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 5. Build Model

In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model   = fasterrcnn_resnet50_fpn_v2(weights=weights)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
model.to(DEVICE)

params    = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
print('Model ready.')

## 6. Train (30 Epochs)

In [ ]:
import time

MODEL_PATH  = '/content/fasterrcnn_best.pth'
best_val    = float('inf')
train_losses, val_losses = [], []
NUM_EPOCHS  = 30

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # Train
    model.train()
    total_train = 0
    for images, targets in train_loader:
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train += loss.item()
    train_loss = total_train / len(train_loader)

    # Val
    total_val = 0
    with torch.no_grad():
        for images, targets in val_loader:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            total_val += sum(loss_dict.values()).item()
    val_loss = total_val / len(val_loader)

    scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    flag = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        flag = ' <- best'

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | {time.time()-t0:.0f}s{flag}')

print('\nTraining complete!')

## 7. Evaluate on Test Set

In [ ]:
import numpy as np

def compute_iou(a, b):
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])
    return inter / (areaA + areaB - inter + 1e-6)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

tp, fp, fn = 0, 0, 0
iou_scores = []

with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(DEVICE) for img in images]
        preds  = model(images)
        for pred, target in zip(preds, targets):
            gt_boxes   = target['boxes'].numpy()
            pred_boxes = pred['boxes'][pred['scores'] >= 0.5].cpu().numpy()
            matched = set()
            for pb in pred_boxes:
                best_iou, best_j = 0, -1
                for j, gb in enumerate(gt_boxes):
                    iou = compute_iou(pb, gb)
                    if iou > best_iou: best_iou, best_j = iou, j
                if best_iou >= 0.5 and best_j not in matched:
                    tp += 1; matched.add(best_j); iou_scores.append(best_iou)
                else:
                    fp += 1
            fn += len(gt_boxes) - len(matched)

precision = tp / (tp + fp + 1e-6)
recall    = tp / (tp + fn + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)
mean_iou  = float(np.mean(iou_scores)) if iou_scores else 0.0

print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Mean IoU  : {mean_iou:.4f}')

## 8. Save Metrics & Download Weights

In [ ]:
import json
from google.colab import files

metrics = {
    'model'    : 'Faster R-CNN (ResNet50-FPN)',
    'precision': round(precision, 4),
    'recall'   : round(recall, 4),
    'f1'       : round(f1, 4),
    'mean_iou' : round(mean_iou, 4),
}

with open('/content/fasterrcnn_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

# Download weights and metrics
files.download('/content/fasterrcnn_best.pth')
files.download('/content/fasterrcnn_metrics.json')